# Pipeline maestro end-to-end TumiPay

Este notebook es el punto único de ejecución de la prueba. Al correrlo de arriba hacia abajo:

1. Conecta con Supabase usando `.env`.
2. Carga los CSV crudos en tablas `raw_*`.
3. Crea la vista `v_clientes_bi` para Power BI.
4. Construye la ABT, entrena el modelo y carga `predicciones_riesgo`.
5. Exporta un mini data lake local en `data/processed/powerbi/` para que Power BI consuma Parquet sin ODBC.
6. Pobla la base vectorial RAG en PostgreSQL/pgvector.
7. Inicializa el agente RAG + LLM y deja una celda editable para hacer preguntas en lenguaje natural.

In [23]:
import sys
import os
import importlib
from pathlib import Path
from dotenv import load_dotenv

current_dir = Path.cwd()
ROOT_DIR = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env", override=True)

import src.config as _cfg_module
importlib.reload(_cfg_module)
from src.config import settings
from src.db_utils import create_supabase_engine, get_database_host, table_counts

engine = create_supabase_engine(settings.DATABASE_URL)
print(f"Conectado a Supabase/PostgreSQL: {get_database_host(settings.DATABASE_URL)}")
print(f"Directorio raw: {settings.DATA_RAW_DIR}")

Conectado a Supabase/PostgreSQL: aws-1-us-east-1.pooler.supabase.com
Directorio raw: C:\Daniel\Mio\project-data-scientist\data\raw


## 1. Cargar datos crudos en Supabase

Carga idempotente desde `data/raw/` hacia `raw_clientes`, `raw_creditos`, `raw_pagos` y `raw_eventos_app`.

In [24]:
from scripts.load_raw_supabase import main as run_carga_raw

run_carga_raw()

raw_tables = ["raw_clientes", "raw_creditos", "raw_pagos", "raw_eventos_app"]
print("\nConteo posterior a carga raw:")
print(table_counts(engine, raw_tables))

2026-05-21 20:02:54,386 - src.ingesta - INFO - Cargando todos los archivos del dataset...
2026-05-21 20:02:54,386 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\clientes.csv
2026-05-21 20:02:54,399 - src.ingesta - INFO - Columna 'fecha_registro' casteada a datetime64[ns].
2026-05-21 20:02:54,400 - src.ingesta - INFO - Archivo clientes.csv cargado exitosamente. Forma: (1400, 16)
2026-05-21 20:02:54,401 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\creditos.csv
2026-05-21 20:02:54,413 - src.ingesta - INFO - Columna 'fecha_desembolso' casteada a datetime64[ns].
2026-05-21 20:02:54,414 - src.ingesta - INFO - Archivo creditos.csv cargado exitosamente. Forma: (1527, 13)
2026-05-21 20:02:54,415 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\pagos.csv
2026-05-21 20:02:54,449 - src.ingesta - INFO - Columna 'fecha_vencimiento' casteada a datetime6

Base de datos activa: aws-1-us-east-1.pooler.supabase.com
raw_clientes: 1400 filas fuente, 1400 filas insertadas/actualizadas
raw_creditos: 1527 filas fuente, 1527 filas insertadas/actualizadas
raw_pagos: 10434 filas fuente, 10434 filas insertadas/actualizadas
raw_eventos_app: 13324 filas fuente, 13324 filas insertadas/actualizadas
          tabla  filas_en_supabase
   raw_clientes               1400
   raw_creditos               1527
      raw_pagos              10434
raw_eventos_app              13324

Conteo posterior a carga raw:
{'raw_clientes': np.int64(1400), 'raw_creditos': np.int64(1527), 'raw_pagos': np.int64(10434), 'raw_eventos_app': np.int64(13324)}


## 2. Crear vista analítica para Power BI

La prueba requiere una capa de visualización. Esta vista prepara el campo de ingreso para BI sin modificar los datos raw: conserva el dato original, capea el valor de presentación al P99 y deja una bandera auditable de outlier.

In [25]:
import pandas as pd
from sqlalchemy import text

sql_view = """
CREATE OR REPLACE VIEW v_clientes_bi AS
WITH p AS (
    SELECT percentile_cont(0.99) WITHIN GROUP (ORDER BY ingreso_mensual_estimado) AS p99
    FROM raw_clientes
    WHERE ingreso_mensual_estimado IS NOT NULL
)
SELECT
    c.*,
    LEAST(c.ingreso_mensual_estimado, (SELECT p99 FROM p)) AS ingreso_mensual_bi,
    CASE
        WHEN c.ingreso_mensual_estimado > (SELECT p99 FROM p) AND COALESCE(c.estrato, 4) <= 3
            THEN 'error_captura'
        WHEN c.ingreso_mensual_estimado > (SELECT p99 FROM p)
            THEN 'outlier_real'
        ELSE 'normal'
    END AS flag_ingreso
FROM raw_clientes c;
"""

with engine.begin() as conn:
    conn.execute(text(sql_view))

print("Vista v_clientes_bi creada/actualizada.")

df_outliers = pd.read_sql("""
SELECT cliente_id, ingreso_mensual_estimado, ingreso_mensual_bi, flag_ingreso, estrato, ocupacion
FROM v_clientes_bi
WHERE flag_ingreso IN ('error_captura', 'outlier_real')
ORDER BY ingreso_mensual_estimado DESC;
""", engine)
display(df_outliers)

Vista v_clientes_bi creada/actualizada.


,cliente_id,ingreso_mensual_estimado,ingreso_mensual_bi,flag_ingreso,estrato,ocupacion
0,CL00493,120000000.0,7152200.0,error_captura,2,Independiente
1,CL00353,10000000.0,7152200.0,outlier_real,4,Empleado
2,CL00686,9450000.0,7152200.0,outlier_real,4,Empleado
3,CL01100,9200000.0,7152200.0,outlier_real,4,Empleado
4,CL00459,9110000.0,7152200.0,outlier_real,5,Empleado
5,CL00258,9000000.0,7152200.0,error_captura,3,Microempresario
6,CL00504,8510000.0,7152200.0,error_captura,2,Empleado
7,CL00676,8260000.0,7152200.0,outlier_real,4,Empleado
8,CL01182,8240000.0,7152200.0,outlier_real,5,Empleado
9,CL01197,7790000.0,7152200.0,outlier_real,5,Empleado


## 3. Entrenar modelo, cargar predicciones y exportar mini data lake

Construye la ABT, entrena LightGBM, serializa `models/modelo_mora.pkl`, hace upsert de `predicciones_riesgo` y exporta tablas Parquet en `data/processed/powerbi/` para que el `.pbix` funcione sin ODBC.

In [30]:
import importlib
import scripts.load_powerbi_predictions as _powerbi_predictions

importlib.reload(_powerbi_predictions)
run_predicciones = _powerbi_predictions.main

run_predicciones()

print("\nConteo posterior a predicciones:")
print(table_counts(engine, ["predicciones_riesgo"]))

powerbi_dir = settings.DATA_PROCESSED_DIR / "powerbi"
print(f"\nMini data lake local para Power BI: {powerbi_dir}")
for path in sorted(powerbi_dir.glob("*.parquet")):
    print(f"- {path.name}")

2026-05-21 20:13:22,858 - src.ingesta - INFO - Cargando todos los archivos del dataset...
2026-05-21 20:13:22,860 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\clientes.csv
2026-05-21 20:13:22,870 - src.ingesta - INFO - Columna 'fecha_registro' casteada a datetime64[ns].
2026-05-21 20:13:22,871 - src.ingesta - INFO - Archivo clientes.csv cargado exitosamente. Forma: (1400, 16)
2026-05-21 20:13:22,871 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\creditos.csv
2026-05-21 20:13:22,878 - src.ingesta - INFO - Columna 'fecha_desembolso' casteada a datetime64[ns].
2026-05-21 20:13:22,879 - src.ingesta - INFO - Archivo creditos.csv cargado exitosamente. Forma: (1527, 13)
2026-05-21 20:13:22,880 - src.ingesta - INFO - Iniciando carga de archivo: C:\Daniel\Mio\project-data-scientist\data\raw\pagos.csv
2026-05-21 20:13:22,910 - src.ingesta - INFO - Columna 'fecha_vencimiento' casteada a datetime6

Base de datos activa: aws-1-us-east-1.pooler.supabase.com


2026-05-21 20:13:23,051 - src.train_mora - INFO - Dataset particionado. Train=1221, Test=306.
2026-05-21 20:13:23,052 - src.train_mora - INFO - Features utilizadas en el modelo (35): ['producto_credito', 'monto_credito', 'plazo_meses', 'tasa_interes_mensual', 'valor_cuota_pactada', 'canal_originacion', 'score_interno_originacion', 'relacion_cuota_ingreso', 'politica_aprobacion', 'departamento', 'ciudad', 'edad', 'genero', 'estrato', 'nivel_educativo', 'ocupacion', 'ingreso_mensual_estimado', 'canal_adquisicion', 'score_externo', 'tiene_producto_ahorro', 'numero_dependientes', 'dispositivo_principal', 'antiguedad_cliente_dias', 'mes_desembolso', 'dia_semana_desembolso', 'prev_evento_actualizacion_datos', 'prev_evento_consulta_saldo', 'prev_evento_login', 'prev_evento_pago_exitoso', 'prev_evento_pago_fallido', 'prev_evento_pago_iniciado', 'prev_evento_simulacion_credito', 'prev_evento_solicitud_soporte', 'prev_evento_sesion_seg_tot', 'prev_evento_sesion_seg_avg']
2026-05-21 20:13:23,053 

[LightGBM] [Info] Number of positive: 333, number of negative: 888
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000682 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2556
[LightGBM] [Info] Number of data points in the train set: 1221, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

## 4. Poblar base vectorial RAG

Indexa en PGVector el resumen del portafolio, políticas de negocio y fichas de créditos. Este paso habilita las preguntas en lenguaje natural del agente.

In [31]:
import importlib
import scripts.populate_rag as _populate_rag

importlib.reload(_populate_rag)
run_populate_rag = _populate_rag.main

chunks_indexados = run_populate_rag(reset=True, include_fichas=True)
print(f"Chunks indexados en PGVector: {chunks_indexados}")

rag_counts = pd.read_sql("""
SELECT
    (SELECT COUNT(*) FROM langchain_pg_collection) AS colecciones,
    (SELECT COUNT(*) FROM langchain_pg_embedding) AS embeddings;
""", engine)
display(rag_counts)

2026-05-21 20:13:34,498 - scripts.populate_rag - INFO - === Iniciando población del RAG Knowledge Base ===
2026-05-21 20:13:34,501 - scripts.populate_rag - INFO - Conectando a PostgreSQL y cargando predicciones_riesgo...


2026-05-21 20:13:36,613 - scripts.populate_rag - INFO -   → 1527 filas cargadas
2026-05-21 20:13:36,618 - scripts.populate_rag - INFO - Construyendo resumen del portafolio...
2026-05-21 20:13:36,680 - scripts.populate_rag - INFO -   → 1 documento(s) de resumen
2026-05-21 20:13:36,683 - scripts.populate_rag - INFO - Construyendo documentos de políticas de negocio...
2026-05-21 20:13:36,685 - scripts.populate_rag - INFO -   → 4 documentos de política
2026-05-21 20:13:36,686 - scripts.populate_rag - INFO - Construyendo fichas de perfil por crédito...
2026-05-21 20:13:36,872 - scripts.populate_rag - INFO -   → 1527 fichas de cliente/crédito
2026-05-21 20:13:36,872 - scripts.populate_rag - INFO - Total documentos antes de chunking: 1532
2026-05-21 20:13:36,872 - scripts.populate_rag - INFO - Total chunks a indexar: 1538
2026-05-21 20:13:36,879 - scripts.populate_rag - INFO - Cargando modelo de embeddings 'intfloat/multilingual-e5-small'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-21 20:13:44,203 - scripts.populate_rag - INFO - Cargando documentos en PGVector (puede tardar varios minutos)...
c:\Daniel\Mio\project-data-scientist\.venv\Lib\site-packages\langchain_community\vectorstores\pgvector.py:490: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use the new operators. 
  store = cls(
2026-05-21 20:15:36,749 - scripts.populate_rag - INFO - RAG Knowledge Base poblado con 1538 chunks.
2026-05-21 20:15:36,751 - scripts.populate_rag - INFO - Verifica con: SELECT COUNT(*) FROM langchain_pg_embedding;


Chunks indexados en PGVector: 1538


,colecciones,embeddings
0,1,1538


## 5. Preguntar al agente RAG + LLM

Edita la variable `pregunta` y ejecuta la celda. El agente recupera contexto desde PGVector y luego llama al LLM de NVIDIA para responder con base en los expedientes indexados.

In [28]:
from src.rag_agent import RAGAgentPipeline

agent = RAGAgentPipeline(
    db_connection=settings.DATABASE_URL,
    nvidia_api_key=settings.NVIDIA_API_KEY,
)

def preguntar_tumipay(pregunta: str) -> str:
    """Consulta el agente RAG y devuelve la respuesta completa del LLM."""
    respuesta = "".join(agent.stream_query(pregunta)).strip()
    print("Pregunta:")
    print(pregunta)
    print("\nRespuesta del agente:")
    print(respuesta)
    return respuesta

# Cambia esta pregunta y vuelve a ejecutar la celda.
pregunta = "¿Cuál es la tasa de mora del portafolio y qué acciones recomiendas para los segmentos de mayor riesgo?"
respuesta = preguntar_tumipay(pregunta)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\Daniel\Mio\project-data-scientist\src\rag_agent.py:45: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use the new operators. 
  self.vector_store = PGVector(


Pregunta:
¿Cuál es la tasa de mora del portafolio y qué acciones recomiendas para los segmentos de mayor riesgo?

Respuesta del agente:
Según el contexto proporcionado, la tasa de mora del portafolio es del 27.2%, lo que se traduce en 416 clientes morosos sobre un total de 1,527 créditos en portafolio.

Para los segmentos de mayor riesgo, recomiendo las siguientes acciones:

1. **Revisión de créditos**: Es importante revisar los créditos de los clientes morosos para identificar las causas subyacentes de la mora. Esto puede ayudar a identificar patrones y tendencias que puedan ser utilizados para mejorar la gestión de la cartera.
2. **Oferta de refinanciación**: Ofrecer refinanciación a los clientes morosos puede ser una forma de ayudarlos a pagar sus deudas y evitar que la situación empeore.
3. **Gestión telefónica**: La gestión telefónica es una herramienta efectiva para comunicarse con los clientes morosos y encontrar soluciones que se adapten a sus necesidades.
4. **Asignación a ges